# 01. EDA（探索的データ分析）

競馬データの基本統計・分布・相関を確認し、回収率に寄与しうる特徴を探索する。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/processed/features.csv', parse_dates=['race_date'])
print(df.shape)
df.head()

In [ ]:
df.describe()

## 1. 着順分布

In [ ]:
fig, axes = plt.subplots(1, 2)
df['finish_pos'].value_counts().sort_index().plot(kind='bar', ax=axes[0], title='着順分布')
df['label_win'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', title='勝/負割合')
plt.tight_layout()
plt.show()

## 2. 人気別 勝率・回収率

In [ ]:
import sys; sys.path.append('../src')
from evaluate_roi import roi_by_popularity

pop_stats = roi_by_popularity(df)
print(pop_stats.head(18))

fig, axes = plt.subplots(1, 2)
pop_stats['hit_rate'][:16].plot(kind='bar', ax=axes[0], title='人気別 勝率 (%)')
pop_stats['roi'][:16].plot(kind='bar', ax=axes[1], title='人気別 単勝回収率 (%)', color='tomato')
for ax in axes:
    ax.set_xlabel('人気')
    ax.axhline(100, color='black', linestyle='--', linewidth=0.8, label='100%')
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. 馬体重変化と着順の関係

In [ ]:
fig, axes = plt.subplots(1, 2)
df.boxplot(column='weight_diff', by='label_win', ax=axes[0])
axes[0].set_title('体重変化 vs 勝負')
axes[0].set_xlabel('勝利 (0=負 / 1=勝)')

df.boxplot(column='weight', by='finish_pos', ax=axes[1])
axes[1].set_title('馬体重 vs 着順')
axes[1].set_xlabel('着順')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 4. 距離・馬場別 勝率・回収率

In [ ]:
from evaluate_roi import roi_by_bet_condition

rows = []
for (surface, dist_cat), grp in df.groupby(['surface', pd.cut(df['distance'], bins=[0,1400,1800,2200,9999],
                                                               labels=['sprint','mile','middle','long'])]):
    r = roi_by_bet_condition(grp, pd.Series(True, index=grp.index))
    r['surface'] = surface
    r['dist_cat'] = dist_cat
    rows.append(r)

dist_df = pd.DataFrame(rows)
pivot = dist_df.pivot(index='dist_cat', columns='surface', values='roi')
pivot.plot(kind='bar', title='馬場×距離区分 単勝回収率 (%)')
plt.axhline(100, color='black', linestyle='--')
plt.tight_layout()
plt.show()
print(pivot)

## 5. 馬場状態別 回収率

In [ ]:
rows = []
for cond, grp in df.groupby('condition'):
    r = roi_by_bet_condition(grp, pd.Series(True, index=grp.index))
    r['condition'] = cond
    rows.append(r)
cond_df = pd.DataFrame(rows).set_index('condition')
cond_df[['hit_rate','roi']].plot(kind='bar', title='馬場状態別 勝率・回収率')
plt.axhline(100, color='gray', linestyle='--')
plt.tight_layout()
plt.show()

## 6. 相関ヒートマップ

In [ ]:
num_cols = ['odds_win','log_odds','popularity','weight','weight_diff',
            'age','distance','condition_code','jockey_te_place','trainer_te_place',
            'finish_pos','label_win','label_place']
corr = df[num_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('特徴量相関ヒートマップ')
plt.tight_layout()
plt.show()

## 7. 騎手別 回収率 Top/Bottom 10

In [ ]:
jockey_roi = []
for j, grp in df.groupby('jockey'):
    if len(grp) < 30:
        continue
    r = roi_by_bet_condition(grp, pd.Series(True, index=grp.index))
    r['jockey'] = j
    jockey_roi.append(r)
jroi_df = pd.DataFrame(jockey_roi).sort_values('roi', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
jroi_df.head(10).set_index('jockey')['roi'].plot(kind='barh', ax=axes[0], title='騎手別回収率 Top10', color='steelblue')
jroi_df.tail(10).set_index('jockey')['roi'].plot(kind='barh', ax=axes[1], title='騎手別回収率 Bottom10', color='salmon')
for ax in axes:
    ax.axvline(100, color='black', linestyle='--')
plt.tight_layout()
plt.show()